<a href="https://colab.research.google.com/github/Chihiro1998/CS5100_FinalProject/blob/main/Dempsey_Center_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/Chihiro1998/CS5800_FinalProject/main/data/"

df22 = pd.read_excel(BASE + "2022%20Annual%20Client%20Survey%20Data.xlsx", header=1)
df23 = pd.read_excel(BASE + "2023%20Annual%20Client%20Survey%20Data.xlsx", header=1)
df24 = pd.read_excel(BASE + "2024%20Annual%20Client%20Survey.xlsx", header=1)
df25 = pd.read_excel(BASE + "Dempsey%20Center%202025%20Annual%20Client%20Survey.xlsx", header=1)

for name, df in [("2022", df22), ("2023", df23), ("2024", df24), ("2025", df25)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols")

2022: 269 rows, 72 cols
2023: 298 rows, 85 cols
2024: 293 rows, 95 cols
2025: 319 rows, 105 cols


In [4]:
# ── Column index mapping ──────────────────────────────────────────
COL = {
    "sat":           {"2022": 0,  "2023": 0,  "2024": 0,  "2025": 4},
    "life_better":   {"2022": 6,  "2023": 1,  "2024": 1,  "2025": 5},
    "nps":           {"2022": 38, "2023": 45, "2024": 57, "2025": 61},
    "freq":          {"2022": 27, "2023": 22, "2024": 34, "2025": 38},
    "age":           {"2022": 47, "2023": 54, "2024": 67, "2025": 74},
    "gender":        {"2022": 49, "2023": 57, "2024": 71, "2025": 78},
    "cancer_impact": {"2022": 43, "2023": 50, "2024": 63, "2025": 68},
}

SAT_MAP = {
    "Very Satisfied":                    5,
    "Satisfied":                         4,
    "Neither Satisfied nor Unsatisfied": 3,
    "Unsatisfied":                       2,
    "Very Unsatisfied":                  1,
}
FREQ_MAP = {
    "More than 10 times": 4,
    "6-9 times":          3,
    "2-5 times":          2,
    "1 time":             1,
}

def extract_year(yr, df):
    n = len(df)
    out = pd.DataFrame({
        # fix: use np.full to properly fill the year column for every row
        "year":          np.full(n, yr),
        "satisfaction":  df.iloc[:, COL["sat"][yr]].map(SAT_MAP).values,
        "life_better":   (df.iloc[:, COL["life_better"][yr]] == "Yes").astype(int).values,
        "nps":           pd.to_numeric(df.iloc[:, COL["nps"][yr]], errors="coerce").values,
        "freq":          df.iloc[:, COL["freq"][yr]].map(FREQ_MAP).values,
        "age":           df.iloc[:, COL["age"][yr]].values,
        "gender":        df.iloc[:, COL["gender"][yr]].values,
        "cancer_impact": df.iloc[:, COL["cancer_impact"][yr]].values,
    })

    out["gender"] = out["gender"].replace({
        "Woman / Female / Feminine":       "Woman",
        "Man / Male / Masculine":          "Man",
        "Non-Binary":                      "Non-Binary/Other",
        "Other / Prefer to self-describe": "Non-Binary/Other",
    })

    if yr == "2025":
        start = pd.to_datetime(df.iloc[:, 2], errors="coerce")
        end   = pd.to_datetime(df.iloc[:, 3], errors="coerce")
        out["duration_min"] = (end - start).dt.total_seconds().values / 60
    else:
        out["duration_min"] = np.nan

    return out

# ── Build combined dataframe ──────────────────────────────────────
raw = {"2022": df22, "2023": df23, "2024": df24, "2025": df25}
clean = pd.concat(
    [extract_year(yr, df) for yr, df in raw.items()],
    ignore_index=True
)

# fix: don't drop_duplicates on the extracted table — people can have
# identical demographic responses without being duplicates
print(f"Total rows after combining: {len(clean)}")

# only drop rows with no satisfaction score (unusable for analysis)
before = len(clean)
clean = clean.dropna(subset=["satisfaction"])
print(f"Rows dropped (no satisfaction score): {before - len(clean)}")
print(f"Final dataset: {len(clean)} rows\n")

# ── Summary table ─────────────────────────────────────────────────
summary = clean.groupby("year").agg(
    n          = ("satisfaction", "count"),
    avg_sat    = ("satisfaction", "mean"),
    pct_better = ("life_better",  "mean"),
    avg_nps    = ("nps",          "mean"),
    nps_n      = ("nps",          "count"),
).round(3)
summary["pct_better"] = (summary["pct_better"] * 100).round(1)

print(summary)

Total rows after combining: 1179
Rows dropped (no satisfaction score): 0
Final dataset: 1179 rows

        n  avg_sat  pct_better  avg_nps  nps_n
year                                          
2022  269    4.539        95.2    9.520    269
2023  298    4.436        95.6    9.629    275
2024  293    4.532        95.9    9.556    266
2025  319    4.542        95.3    9.616    292
